# 05 — Four-Queue Hawkes System (Sections 1.2.4-5)

Complete answers for:
- **Q1.2.4.5**: Four-model hitting time comparison
- **Q1.2.5.1**: Second limit conditional distributions (constant rates)
- **Q1.2.5.2**: Second limit with Hawkes cross-excitation (eq. 4)
- **Interaction diagram**: How 8 processes influence each other

---

In [ ]:
import sys; sys.path.insert(0, '.')
import numpy as np
import matplotlib.pyplot as plt
from model.hawkes_4q import *
from model.hawkes import simulate_hawkes_queue, simulate_coupled_hawkes
from model.hitting_times import simulate_until_hit_zero, batch_hitting_times

plt.rcParams.update({'figure.figsize': (12, 4), 'font.size': 11,
                      'axes.grid': True, 'grid.alpha': 0.3})

## 0. Interaction Diagram

The 4-queue system has 8 processes. Each arrow shows how one process's events affect another's intensity.

In [ ]:
fig, ax = plt.subplots(figsize=(14, 10))
ax.set_xlim(-1, 11)
ax.set_ylim(-1, 9)
ax.set_aspect('equal')
ax.axis('off')

# ── Queue boxes ──────────────────────────────────────────────────
box_style = dict(boxstyle='round,pad=0.4', facecolor='white', edgecolor='black', lw=1.5)
bid_color = '#B5D4F4'
ask_color = '#F5C4B3'

boxes = {
    'Q+1': (2, 7, bid_color, '$Q^{+1}$ (bid 1st)'),
    'Q-1': (8, 7, ask_color, '$Q^{-1}$ (ask 1st)'),
    'Q+2': (2, 2, bid_color, '$Q^{+2}$ (bid 2nd)'),
    'Q-2': (8, 2, ask_color, '$Q^{-2}$ (ask 2nd)'),
}

# Draw rate labels inside each box
for name, (x, y, color, label) in boxes.items():
    rect = plt.Rectangle((x-0.9, y-0.7), 1.8, 1.4, facecolor=color,
                          edgecolor='black', lw=1.5, zorder=2, alpha=0.8)
    ax.add_patch(rect)
    ax.text(x, y+0.3, label, ha='center', va='center', fontsize=11, fontweight='bold', zorder=3)
    if '+1' in name or '-1' in name:
        ax.text(x, y-0.2, '$\\lambda^+$=const, $\\lambda^-$=Hawkes', ha='center',
                va='center', fontsize=8, color='#444', zorder=3)
    else:
        ax.text(x, y-0.2, '$\\lambda^+$=Hawkes*, $\\lambda^-$=const', ha='center',
                va='center', fontsize=8, color='#444', zorder=3)

# ── Arrows ────────────────────────────────────────────────────────
arrow_kw = dict(arrowstyle='->', lw=1.5, mutation_scale=15)

def draw_arrow(ax, x1, y1, x2, y2, color, label, offset=(0,0), curved=0):
    style = f'arc3,rad={curved}' if curved else 'arc3,rad=0'
    ax.annotate('', xy=(x2+offset[0], y2+offset[1]),
                xytext=(x1+offset[0], y1+offset[1]),
                arrowprops=dict(arrowstyle='->', color=color, lw=1.8,
                                connectionstyle=style))
    mx = (x1+x2)/2 + offset[0] + curved*1.5
    my = (y1+y2)/2 + offset[1] + abs(curved)*0.3
    ax.text(mx, my, label, fontsize=8, color=color, ha='center', va='center',
            bbox=dict(facecolor='white', edgecolor='none', alpha=0.8, pad=1))

# Self-interactions (loops) for first limits
for x, name in [(2, '+1'), (8, '-1')]:
    # Self-excitation loop
    ax.annotate('', xy=(x+0.9, 7.5), xytext=(x+0.9, 7.8),
                arrowprops=dict(arrowstyle='->', color='#E24B4A', lw=1.5,
                                connectionstyle='arc3,rad=-1.5'))
    ax.text(x+1.7, 8.2, 'self\n$dN^- {\\to} \\lambda^- \\uparrow$\n$dN^+ {\\to} \\lambda^- \\downarrow$',
            fontsize=7, color='#E24B4A', ha='center', va='center')

# Cross-interaction: Q+1 ↔ Q-1 (mutual excitation of λ⁻)
draw_arrow(ax, 3.0, 7.4, 7.0, 7.4, '#534AB7',
           '$dN^{+1,\\pm} \\to \\lambda^{-1,-} \\uparrow$', curved=0.3)
draw_arrow(ax, 7.0, 6.6, 3.0, 6.6, '#534AB7',
           '$dN^{-1,\\pm} \\to \\lambda^{+1,-} \\uparrow$', curved=0.3)

# Cross Q+1 → Q+2 (eq. 4: first limit removals excite second limit additions)
draw_arrow(ax, 2.0, 6.3, 2.0, 2.8, '#1D9E75',
           '$dN^{+1,-} \\to \\lambda^{+2,+} \\uparrow$  (eq.4)', offset=(-0.5, 0))

# Cross Q-1 → Q-2
draw_arrow(ax, 8.0, 6.3, 8.0, 2.8, '#1D9E75',
           '$dN^{-1,-} \\to \\lambda^{-2,+} \\uparrow$  (eq.4)', offset=(0.5, 0))

# Legend
ax.text(5, -0.5, 'Purple: cross-excitation between first limits (Hawkes, eq. 3)\n'
        'Green: first-limit removals excite second-limit additions (eq. 4)\n'
        'Red: self-interaction within each first limit (Hawkes, eq. 1)',
        ha='center', va='center', fontsize=9, color='#444',
        bbox=dict(facecolor='#F1EFE8', edgecolor='#D3D1C7', pad=6, boxstyle='round'))

ax.set_title('Interaction diagram: 4 queues, 8 processes', fontsize=14, pad=20)
plt.tight_layout()
plt.show()

## 1. Q1.2.4.5 — Four-model hitting time comparison

Compare:
1. Single Poisson queue
2. min(two independent Poisson)
3. Single Hawkes queue
4. min(two coupled Hawkes)

Parameters: $\mu^+ = 1.2$, $\mu^- = 1.5$, $\alpha = 0.3$, $\beta = 0.5$, $Q(0) = 10$.

In [ ]:
comp = four_model_comparison(n_runs=500, q_init=10,
                              mu_plus=1.2, mu_minus=1.5,
                              alpha=0.3, beta=0.5, seed=42)

print(f'{"Model":<22s} {"E[T] (ms)":>10s} {"± 95%CI":>10s} {"n":>6s}')
print('─' * 52)
for name in ['single_poisson', 'two_poisson', 'single_hawkes', 'two_hawkes']:
    s = comp['summary'][name]
    print(f'{name:<22s} {s["mean"]:10.1f} {s["ci_95"]:10.1f} {s["n"]:6d}')

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(15, 5))

# Left: histograms
ax = axes[0]
colors = {'single_poisson': 'C0', 'two_poisson': 'C1',
          'single_hawkes': 'C3', 'two_hawkes': 'C4'}
labels = {'single_poisson': '1 Poisson', 'two_poisson': 'min(2 Poisson)',
          'single_hawkes': '1 Hawkes', 'two_hawkes': 'min(2 Hawkes)'}
t_max = 120
bins = np.linspace(0, t_max, 40)
for name in ['single_poisson', 'two_poisson', 'single_hawkes', 'two_hawkes']:
    arr = comp['raw'][name]
    arr = arr[arr < t_max]
    ax.hist(arr, bins=bins, density=True, alpha=0.4,
            color=colors[name], label=labels[name], edgecolor='white')
ax.set_xlabel('Hitting time (ms)')
ax.set_ylabel('Density')
ax.set_title('Hitting time distributions')
ax.legend(fontsize=9)

# Right: bar chart of means
ax = axes[1]
names_ord = ['two_poisson', 'two_hawkes', 'single_poisson', 'single_hawkes']
means = [comp['summary'][n]['mean'] for n in names_ord]
cis = [comp['summary'][n]['ci_95'] for n in names_ord]
bar_labels = ['min(2 Poi)', 'min(2 Hawk)', '1 Poisson', '1 Hawkes']
bar_colors = [colors[n] for n in names_ord]
bars = ax.bar(range(4), means, yerr=cis, capsize=5,
              color=bar_colors, edgecolor='white', width=0.6)
ax.set_xticks(range(4))
ax.set_xticklabels(bar_labels, fontsize=10)
ax.set_ylabel('E[T] (ms)')
ax.set_title('Mean hitting time comparison')
for i, (m, c) in enumerate(zip(means, cis)):
    ax.text(i, m + c + 1, f'{m:.1f}', ha='center', fontsize=10, fontweight=500)

plt.tight_layout()
plt.show()

### Discussion: the v4 sign convention creates inertia

With the corrected signs $\lambda^-(t) = \mu^- - \int \phi\, dN^+ + \int \phi\, dN^-$:
- When the queue **grows** (adds > removes): $\lambda^-$ **decreases** → removals slow down → queue keeps growing
- When the queue **shrinks**: $\lambda^-$ **increases** → removals speed up → queue keeps shrinking

This **inertia** makes it harder to deplete the queue from a stable state (it resists random fluctuations),  
resulting in **longer** hitting times compared to Poisson.

Ordering observed: min(2 Poi) < min(2 Hawkes) < 1 Poisson < 1 Hawkes.

## 2. Four-queue simulation: trajectories

In [ ]:
# ── Run a long 4-queue sim without stopping ──────────────────────
p_base = FourQueueParams(mu_plus_1=1.2, mu_minus_1=1.5,
                          alpha=0.3, beta=0.5,
                          mu_plus_2=0.8, mu_minus_2=0.6,
                          q1_init=10, q_neg1_init=10,
                          q2_init=5, q_neg2_init=5,
                          a_cross=0.0)

res_long = simulate_4queue(p_base, T_max=100,
                            rng=np.random.default_rng(123),
                            stop_at_first_limit_zero=False)

fig, axes = plt.subplots(2, 2, figsize=(15, 9))

# Queue trajectories
for ax, qi, label, color in [
    (axes[0,0], 1,  '$Q^{+1}$ (bid 1st)', 'C0'),
    (axes[0,1], -1, '$Q^{-1}$ (ask 1st)', 'C3'),
    (axes[1,0], 2,  '$Q^{+2}$ (bid 2nd)', 'C0'),
    (axes[1,1], -2, '$Q^{-2}$ (ask 2nd)', 'C3'),
]:
    ax.step(res_long.times, res_long.q_paths[qi], where='post', lw=0.6, color=color)
    ax.set_ylabel('Queue size')
    ax.set_title(label)

for ax in axes[1]:
    ax.set_xlabel('Time (ms)')
plt.suptitle('4-queue trajectories (no cross-excitation, a=0)', fontsize=13, y=1.01)
plt.tight_layout()
plt.show()

## 3. Q1.2.5.1 — Conditional distributions (constant 2nd limit rates)

What is the distribution of $N^{+2}_t$ when the first limit hits zero?  
**(a)** when $N^{+1}_t = 0$ (same side depleted)  
**(b)** when $N^{-1}_t = 0$ (opposite side depleted)

In [ ]:
p_const = FourQueueParams(a_cross=0.0, q2_init=5, q_neg2_init=5)

print('Computing conditional distributions (a_cross=0, n=1000)...')
cd_const = conditional_q2_at_depletion(p_const, n_runs=1000, seed=42)

print(f'  Q2 same side when 1st depleted: mean={cd_const["q2_same"].mean():.2f}, '
      f'std={cd_const["q2_same"].std():.2f}')
print(f'  Q2 opp side when 1st depleted:  mean={cd_const["q2_opp"].mean():.2f}, '
      f'std={cd_const["q2_opp"].std():.2f}')
print(f'  ({cd_const["n_valid"]} valid runs)')

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

max_q = int(max(cd_const['q2_same'].max(), cd_const['q2_opp'].max())) + 2
bins_q = np.arange(-0.5, max_q + 0.5, 1)

ax = axes[0]
ax.hist(cd_const['q2_same'], bins=bins_q, density=True, alpha=0.7,
        color='C0', edgecolor='white')
ax.axvline(cd_const['q2_same'].mean(), ls='--', color='black', lw=1.5,
           label=f'mean = {cd_const["q2_same"].mean():.1f}')
ax.set_xlabel('$Q^{+2}$ at depletion')
ax.set_ylabel('Density')
ax.set_title('(a) $Q^{+2}$ when same-side $Q^{+1} = 0$')
ax.legend()

ax = axes[1]
ax.hist(cd_const['q2_opp'], bins=bins_q, density=True, alpha=0.7,
        color='C3', edgecolor='white')
ax.axvline(cd_const['q2_opp'].mean(), ls='--', color='black', lw=1.5,
           label=f'mean = {cd_const["q2_opp"].mean():.1f}')
ax.set_xlabel('$Q^{+2}$ at depletion')
ax.set_ylabel('Density')
ax.set_title('(b) $Q^{+2}$ when opposite $Q^{-1} = 0$')
ax.legend()

plt.suptitle('Q1.2.5.1: Second limit distribution at first-limit depletion (a=0)', fontsize=13)
plt.tight_layout()
plt.show()

## 4. Q1.2.5.2 — Cross-excitation: sensitivity to parameter $a$

Equation (4): $\lambda^{-2,+}(t) = \mu^{-2,+} + \int a\, e^{-b(t-s)} dN^{-1,-}$

When the first limit is being depleted (many $dN^{-1,-}$ events),  
the second limit's **addition rate increases** — people rush to queue at the second limit.

In [ ]:
# ── Scan over a_cross values ──────────────────────────────────────
a_values = [0.0, 0.2, 0.5, 1.0, 2.0]
results_by_a = {}

for a_val in a_values:
    p = FourQueueParams(a_cross=a_val, q2_init=5, q_neg2_init=5)
    cd = conditional_q2_at_depletion(p, n_runs=500, seed=42)
    results_by_a[a_val] = cd
    print(f'a={a_val:.1f}: Q2_same={cd["q2_same"].mean():.1f}, '
          f'Q2_opp={cd["q2_opp"].mean():.1f} ({cd["n_valid"]} runs)')

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

# Left: Q2 same-side distributions for different a
ax = axes[0]
for a_val in a_values:
    cd = results_by_a[a_val]
    ax.hist(cd['q2_same'], bins=np.arange(-0.5, 30.5, 1), density=True,
            alpha=0.4, label=f'a = {a_val}')
ax.set_xlabel('$Q^{+2}$')
ax.set_ylabel('Density')
ax.set_title('(a) $Q^{+2}$ when same-side $Q^{+1}=0$')
ax.legend(fontsize=9)

# Middle: Q2 opposite-side
ax = axes[1]
for a_val in a_values:
    cd = results_by_a[a_val]
    ax.hist(cd['q2_opp'], bins=np.arange(-0.5, 30.5, 1), density=True,
            alpha=0.4, label=f'a = {a_val}')
ax.set_xlabel('$Q^{+2}$')
ax.set_ylabel('Density')
ax.set_title('(b) $Q^{+2}$ when opposite $Q^{-1}=0$')
ax.legend(fontsize=9)

# Right: mean Q2 as function of a
ax = axes[2]
means_same = [results_by_a[a]['q2_same'].mean() for a in a_values]
means_opp = [results_by_a[a]['q2_opp'].mean() for a in a_values]
ax.plot(a_values, means_same, 'o-', lw=2, ms=6, color='C0',
        label='Same side ($Q^{+1}=0$)')
ax.plot(a_values, means_opp, 's-', lw=2, ms=6, color='C3',
        label='Opposite ($Q^{-1}=0$)')
ax.set_xlabel('Cross-excitation parameter $a$')
ax.set_ylabel('Mean $Q^{+2}$ at depletion')
ax.set_title('Sensitivity: mean 2nd limit size at depletion')
ax.legend()

plt.suptitle('Q1.2.5.2: Effect of cross-excitation (eq. 4) on second limit', fontsize=13)
plt.tight_layout()
plt.show()

print('Key finding: as a increases, the second limit is LARGER when the first is depleted.')
print('This models the "rush to queue" effect: participants anticipate promotion of Q2 to Q1.')

## 5. Comparison with and without cross-excitation: 4-queue trajectories

In [ ]:
# ── Side by side: a=0 vs a=1.0 ───────────────────────────────────
fig, axes = plt.subplots(2, 2, figsize=(15, 9))

for col, a_val, title_suffix in [(0, 0.0, 'a=0 (no excitation)'),
                                   (1, 1.0, 'a=1.0 (strong excitation)')]:
    p = FourQueueParams(a_cross=a_val, q1_init=10, q_neg1_init=10,
                        q2_init=5, q_neg2_init=5)
    res = simulate_4queue(p, T_max=100,
                           rng=np.random.default_rng(42),
                           stop_at_first_limit_zero=False)

    for row, (qi, label, color) in enumerate([
        (1, '$Q^{+1}$ (bid 1st)', 'C0'),
        (2, '$Q^{+2}$ (bid 2nd)', 'C2'),
    ]):
        ax = axes[row, col]
        ax.step(res.times, res.q_paths[qi], where='post', lw=0.7, color=color)
        ax.set_ylabel(label)
        if row == 0:
            ax.set_title(title_suffix)
        if row == 1:
            ax.set_xlabel('Time (ms)')

plt.suptitle('Effect of cross-excitation on second limit dynamics', fontsize=13, y=1.01)
plt.tight_layout()
plt.show()

print('With a=1.0, the second limit grows MUCH faster when the first is being depleted.')
print('This ensures more liquidity is available when Q2 is promoted to best limit.')

## 6. Hawkes intensity trajectories in the 4-queue system

In [ ]:
p_show = FourQueueParams(a_cross=0.5)
res_show = simulate_4queue(p_show, T_max=80,
                            rng=np.random.default_rng(42),
                            stop_at_first_limit_zero=False)

fig, axes = plt.subplots(2, 2, figsize=(15, 8))

# Top: queue sizes
ax = axes[0, 0]
ax.step(res_show.times, res_show.q_paths[1], where='post', lw=0.7, color='C0', label='$Q^{+1}$')
ax.step(res_show.times, res_show.q_paths[-1], where='post', lw=0.7, color='C3', label='$Q^{-1}$')
ax.set_ylabel('Queue size')
ax.set_title('First limits')
ax.legend(fontsize=9)

ax = axes[0, 1]
ax.step(res_show.times, res_show.q_paths[2], where='post', lw=0.7, color='C0', label='$Q^{+2}$')
ax.step(res_show.times, res_show.q_paths[-2], where='post', lw=0.7, color='C3', label='$Q^{-2}$')
ax.set_ylabel('Queue size')
ax.set_title('Second limits')
ax.legend(fontsize=9)

# Bottom: intensities
ax = axes[1, 0]
ax.step(res_show.times, res_show.lam_minus_paths[1], where='post', lw=0.5, color='C0', label='$\\lambda^{+1,-}$')
ax.step(res_show.times, res_show.lam_minus_paths[-1], where='post', lw=0.5, color='C3', label='$\\lambda^{-1,-}$')
ax.axhline(p_show.stationary_lam_minus, ls='--', color='black', lw=1, label='$m^-$ theory')
ax.set_xlabel('Time (ms)')
ax.set_ylabel('$\\lambda^-$')
ax.set_title('First limit removal rates (Hawkes)')
ax.legend(fontsize=9)

ax = axes[1, 1]
ax.step(res_show.times, res_show.lam_plus_2_paths[2], where='post', lw=0.5, color='C0', label='$\\lambda^{+2,+}$')
ax.step(res_show.times, res_show.lam_plus_2_paths[-2], where='post', lw=0.5, color='C3', label='$\\lambda^{-2,+}$')
ax.axhline(p_show.mu_plus_2, ls='--', color='black', lw=1, label=f'baseline $\\mu^{{+2,+}}$ = {p_show.mu_plus_2}')
ax.set_xlabel('Time (ms)')
ax.set_ylabel('$\\lambda^+$ (2nd limit)')
ax.set_title('Second limit addition rates (cross-excited, eq. 4)')
ax.legend(fontsize=9)

plt.suptitle(f'4-queue system: all intensities (a_cross={p_show.a_cross})', fontsize=13, y=1.01)
plt.tight_layout()
plt.show()

---
## Summary

| Question | Key Finding |
|----------|-------------|
| Q1.2.4.5 | With v4 signs (inertia), Hawkes has LONGER hitting times than Poisson. Ordering: min(2P) < min(2H) < 1P < 1H. |
| Q1.2.5.1a | When same-side 1st limit depleted: 2nd limit is near its initial value (constant rates, no coupling). |
| Q1.2.5.1b | When opposite 1st limit depleted: similar distribution (symmetric problem). |
| Q1.2.5.2 | Cross-excitation (a > 0) makes the 2nd limit **grow** as the 1st is depleted — the "rush to queue" effect. |
| Sensitivity | Mean 2nd limit at depletion scales roughly linearly with parameter $a$. |